In [ ]:
import os
import warnings
import random
import pickle
from datetime import datetime
from collections import defaultdict
from functools import partial

# Environment configuration
os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=8'

import numpyro
from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS, log_likelihood, hmc
from numpyro.infer.util import initialize_model
from numpyro.util import fori_collect

import jax
from jax import config
config.update("jax_enable_x64", True)
config.update('jax_platform_name', 'cpu')

import jax.numpy as jnp
from jax import jit, pmap, devices, device_get, lax, local_device_count, random, vmap, block_until_ready
from jax.random import PRNGKey, split
from jax.scipy.optimize import minimize
from jax.scipy.signal import fftconvolve
from jax.scipy.signal import convolve as jax_convolve

# Check devices
print('Available devices:', devices())
print('CPU devices:', devices('cpu'))

# Suppress warnings
warnings.filterwarnings('ignore', message="It appears that you're using a Mac with one of Apple's ARM-based processors")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, to_hex
import corner

import astropy.units as u
import astropy.constants as c
from astropy.constants import G, m_p
from astropy.table import Table

import shone

import fleck
from fleck.jax import ActiveStar

from scipy.optimize import fmin_powell, curve_fit
from scipy.stats import gaussian_kde

from chromatic import *
from svo_filters import svo
from bt_settl import get_interp_stellar_spectrum
from bt_settl_3d import get_interp_stellar_spectrum_3d

import arviz
import arviz as az

from tqdm.auto import tqdm
from glob import glob

# Create custom binned bin edges
wl_min, wl_max = 0.7, 1.7
n_points = 10000

binned_bin_edges = np.geomspace(wl_min, wl_max, n_points + 1)

binned_wavelengths = (binned_bin_edges[:-1] + binned_bin_edges[1:]) / 2.0

print(f"Creating custom binned grid with {n_points} points from {wl_min} to {wl_max} µm")
print(f"Binned bin edges shape: {len(binned_bin_edges)}")
print(f"Binned wavelengths shape: {len(binned_wavelengths)}")

# Step 2: Create the interpolation grid using these custom bin edges
print("\nCreating BT-Settl 3D interpolation grid with custom binning...")
panchromatic_btsettl_grid = get_interp_stellar_spectrum_3d(binned_bin_edges)
panchromatic_wavelengths = binned_wavelengths
panchromatic_bin_edges = binned_bin_edges

print(f"\nWavelength range: {panchromatic_wavelengths[0]:.3f} - {panchromatic_wavelengths[-1]:.3f} µm")
print(f"Number of wavelength points: {len(panchromatic_wavelengths)}")
print("\nGrid creation complete!")

In [ ]:
visits = {
    'F21': {
        'Grism': 'G141',
        # 'Forward': G_141_for_dict,
        # 'Backward': G_141_back_dict,
        'BJD_times': np.array(pd.read_csv('../../data/F21_bjdtimes.csv')['BJD'][:]) * u.day,
        'time_lower': 2459455.708 * u.day,
        'time_upper': 2459455.738 * u.day,
        'T0 (BJD_TDB)': 2459455.9895 * u.day,
        'exp (s)': 4.9784 * u.s,
        'native resolution': 46.3 * u.angstrom
    },
    'S22': {
        'Grism': 'G102',
        # 'Forward': G_102_for_dict,
        # 'Backward': G_102_back_dict,
        'BJD_times': np.array(pd.read_csv('../../data/S22_bjdtimes.csv')['BJD'][:]) * u.day,
        'time_lower': 2459684.215 * u.day,
        'time_upper': 2459684.243 * u.day,
        'T0 (BJD_TDB)': 2459684.4959 * u.day, # This is 27 planetary orbits after the first transit, + 0.0054 days (the transit arrived 7 minutes late)
        'exp (s)': 9.67632 * u.s,
        'native resolution': 24.6 * u.angstrom
    }
}

systeminfo = {
    'duration (hr)': 3.5 * u.hr,
    'T_orb (d)': 8.463 * u.day,
    'T_rot (d)': 4.86 * u.day,
    'inclination': 89.5,
    'eccentricity': 0.0,
    'longitude_of_periastron': 88.4
}

def read_sensitivity_curve(grism='G141'):
    path = f'../../data/WFC3.IR.{grism}.1st.sens.2.fits'

    response = fits.open(path)

    w = response[1].data['wavelength']/1e4 * u.micron
    s = response[1].data['sensitivity'] * u.cm * u.cm / u.erg
    e = response[1].data['error'] * u.cm * u.cm / u.erg
    
    return w, s, e

@jit
def get_planck_spectrum_jax(T, **kwargs):
    """
    Calculate the surface flux from a thermally emitted surface,
    according to Planck function.

    Parameters
    ----------
    wavelength : Quantity
        The wavelengths at which to calculate,
        with units of wavelength.
    temperature : Quantity
        The temperature of the thermal emitter,
        with units of K.

    Returns
    -------
    surface_flux : Quantity
        The surface flux, evaluated at the wavelengths.
    """

    # define variables as shortcut to the constants we need
    h = 6.62607e-27 # erg s
    k = 1.380649e-16 # erg/K
    c = 2.9979e18 # angstrom/s
    wavelength = panchromatic_wavelengths*1e4

    z = h * c / (wavelength * k * T) # units check out

    # calculate the intensity from the Planck function
    intensity = (2 * h * c**2 / wavelength**5 / (jnp.exp(z) - 1)) # Units are erg/s/A^3

    # calculate the flux assuming isotropic emission
    flux = jnp.pi * intensity * 1e16 # erg / (s * cm^2 * angstrom)

    # return the intensity
    wave_jax = jnp.array(panchromatic_wavelengths)
    flux_jax = jnp.array(flux)

    return wave_jax, flux_jax



@jit
def get_BTSettl_spectrum_jax(T, 
                             metallicity, 
                             grid=panchromatic_btsettl_grid, **kwargs):
    """
    Get BT-Settl spectrum for given temperature, logg, and metallicity.
    
    Parameters:
    T : effective temperature (K)
    logg : surface gravity (log10(cm/s^2))
    metallicity : metallicity ([M/H])
    grid : 3D interpolation function from get_interp_stellar_spectrum_3d
    
    Returns:
    wave_jax : wavelength array (micrometers)
    flux_jax : flux array (erg/cm^2/s/Å)
    """
    
    # Pass all three parameters to the 3D grid interpolation
    gridspec = grid(
        jnp.array(T, dtype=jnp.float64),
        # jnp.array(logg, dtype=jnp.float32),
        jnp.array(metallicity, dtype=jnp.float64)
    )
    
    sigma_sb = 5.67e-5  # erg/cm^2/s/K^4
    nf = (sigma_sb * (T)**4) / (jnp.trapezoid(gridspec, x=jnp.array(panchromatic_wavelengths)*1e4))
    re_normed_flux = gridspec * nf
    
    wave_jax = jnp.array(panchromatic_wavelengths)
    flux_jax = jnp.array(re_normed_flux)
    
    return wave_jax, flux_jax

In [ ]:
###############################
# S22
###############################

visit = 'S22'
'Things that automatically get re-defined for either visit'
predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
binwidth = visits[f'{visit}']['native resolution']
exptime = visits[f'{visit}']['exp (s)']
grism = visits[f'{visit}']['Grism']
rainbow = read_rainbow(f"../../data/rainbows/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")
speclc_rainbow = rainbow
'remove the transit'
out_of_transit = ( speclc_rainbow.trim().mask_transit(period=8.463*u.day, t0=predicted_T0*u.day, duration=0.15*u.day) )
# out_of_transit.imshow()
'define the appropriate arrays'
_meanOOTspec = out_of_transit.get_average_spectrum_as_rainbow()
meanOOTspec = _meanOOTspec.flux
meanOOTspec_relative_err = _meanOOTspec.uncertainty.value/meanOOTspec.value
oot_spec_err = meanOOTspec_relative_err.flatten()
_SED_wavelengths = (_meanOOTspec.wavelength.value).flatten()
e_per_s = meanOOTspec / exptime
e_per_s_per_angstrom = e_per_s / binwidth
_w, _s, _e = read_sensitivity_curve(grism=grism)
binned_filter_response = bintogrid(_w.value, _s.value, newx=_SED_wavelengths)['y'] * u.cm**2 / u.erg
_calibrated_mean_spec = e_per_s_per_angstrom.flatten() / binned_filter_response
calibrated_mean_spec = _calibrated_mean_spec.value/np.nanmean(_calibrated_mean_spec.value)
oot_spec_err = oot_spec_err * calibrated_mean_spec
# Convert to JAX arrays
calibrated_mean_spec_G102 = jnp.array(calibrated_mean_spec)
oot_spec_err_G102 = jnp.array(oot_spec_err)
SED_wavelengths_G102 = jnp.array(_SED_wavelengths)

###############################
# F21
###############################
visit = 'F21'
'Things that automatically get re-defined for either visit'
predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
binwidth = visits[f'{visit}']['native resolution']
exptime = visits[f'{visit}']['exp (s)']
grism = visits[f'{visit}']['Grism']
rainbow = read_rainbow(f"../../data/rainbows/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")
speclc_rainbow = rainbow
'remove the transit'
out_of_transit = ( speclc_rainbow.trim().mask_transit(period=8.463*u.day, t0=predicted_T0*u.day, duration=0.15*u.day) )
'define the appropriate arrays'
_meanOOTspec = out_of_transit.get_average_spectrum_as_rainbow()
meanOOTspec = _meanOOTspec.flux
meanOOTspec_relative_err = _meanOOTspec.uncertainty.value/meanOOTspec.value
_oot_spec_err = meanOOTspec_relative_err.flatten()
_SED_wavelengths = (_meanOOTspec.wavelength.value).flatten()
e_per_s = meanOOTspec / exptime
e_per_s_per_angstrom = e_per_s / binwidth
_w, _s, _e = read_sensitivity_curve(grism=grism)
binned_filter_response = bintogrid(_w.value, _s.value, newx=_SED_wavelengths)['y'] * u.cm**2 / u.erg
_calibrated_mean_spec = e_per_s_per_angstrom.flatten() / binned_filter_response
calibrated_mean_spec = _calibrated_mean_spec.value/np.nanmean(_calibrated_mean_spec.value)
oot_spec_err = _oot_spec_err * calibrated_mean_spec
# Convert to JAX arrays
calibrated_mean_spec_G141 = jnp.array(calibrated_mean_spec)
oot_spec_err_G141 = jnp.array(oot_spec_err)
SED_wavelengths_G141 = jnp.array(_SED_wavelengths)

###############################
# PLOT DATA
###############################
plt.figure()
plt.errorbar(SED_wavelengths_G102,calibrated_mean_spec_G102,yerr=oot_spec_err_G102*100,fmt='o',label='G102',ms=1)
plt.errorbar(SED_wavelengths_G141,calibrated_mean_spec_G141,yerr=oot_spec_err_G141*100,fmt='o',label='G141',ms=1)
plt.legend()
plt.show()
plt.clf()

In [ ]:
@jit
def convolve_spectrum_jax(model_wavelength, model_flux, sigma, kernel_size=7, **kwargs):
    """
    Properly convolve a spectrum with a Gaussian kernel in JAX.
    
    Args:
        model_wavelength: Array of wavelengths (must be evenly spaced!)
        model_flux: Corresponding flux values
        sigma: Standard deviation of Gaussian kernel in wavelength units
        kernel_size: Number of elements in the kernel (odd number recommended)
        
    Returns:
        Convolved flux array
    """
    # Ensure inputs are JAX arrays
    model_wavelength = jnp.asarray(model_wavelength)
    model_flux = jnp.asarray(model_flux)
    
    # Create proper Gaussian kernel
    x = jnp.linspace(-(kernel_size//2), kernel_size//2, kernel_size)
    kernel = jnp.exp(-0.5 * (x/sigma)**2)
    kernel = kernel / jnp.sum(kernel)  # normalize
    
    # Perform convolution
    convolved = jax_convolve(model_flux, kernel, mode='same', method='fft')
    
    return convolved

In [ ]:
N_temps = 2
'Gridpoint Params'
metallicity = 0.0 #numpyro.sample('metallicity', dist.Uniform(-1.0, 0.5))
# logg = 4.5 #numpyro.sample('logg', dist.Uniform(4.0, 5.0))
sigma_conv = 0.6
combined_SED_wavelengths = jnp.concatenate([jnp.array(SED_wavelengths_G102), 
                                            jnp.array(SED_wavelengths_G141)])

def decomp_model(temps, scales, metal_val, sigma):
    # Get spectra for all temperatures with variable metallicity AND logg
    spectra = [get_BTSettl_spectrum_jax(T=Temp, metallicity=metal_val) 
                for Temp in temps]
    
    # Calculate combined spectrum with scaling factors
    flux_combined = scales[0] * spectra[0][1]  # Start with first component
    for i in range(1, N_temps):
        flux_combined += scales[i] * spectra[i][1]
    
    # Normalize
    _model_flux = flux_combined / jnp.mean(flux_combined)
    
    _binned_model_flux = shone.bin_spectrum(
        combined_SED_wavelengths, 
        panchromatic_wavelengths, 
        _model_flux
    )
    # Convolve and bin
    binned_model_flux = convolve_spectrum_jax(
        panchromatic_wavelengths, 
        _binned_model_flux, 
        sigma=jnp.asarray(sigma, dtype=jnp.float64)
    )

    model_flux = binned_model_flux / jnp.mean(binned_model_flux)
    
    return model_flux[2:-2]

# Calculate model flux with both metallicity AND logg
model_flux = decomp_model([3800,3100], [1.0,1.0], metallicity, sigma_conv)

plt.plot(combined_SED_wavelengths[2:-2], model_flux)

# 1T BT-SETTL Modeling

# 2T BT-SETTL Modeling

In [ ]:
def residual(parameters, sigma, data_list):

    data_wave, data_flux, data_err = data_list
    
    _model = spectral_model(parameters, sigma)
    binned_model = bintogrid(btsettl_wavelengths, _model, newx = data_wave.value)
    model_flux = binned_model['y'] * u.erg / u.s / u.angstrom / u.cm / u.cm

    residual = (data_flux-model_flux)/data_err
    
    return residual
    
# Define the log-probability function
def log_probability(parameters, data_wave, data_flux, data_err):

    f_cool, T_cool, T_phot, R_star, conv_sigma = parameters

    log_likelihood = 0
    
    # Boundary conditions for parameters to ensure physical validity
    if not (0.05<=f_cool<=0.95 and 1800 <= T_cool <= T_phot <= 6000 and 0.5 <= R_star <= 0.90 and 1.0 <= conv_sigma <= 40.0):
        return -np.inf  # Return log-probability of -inf outside bounds

    # Set parameters in lmfit
    model_params = lmfit.Parameters()
    model_params.add('f_cool', value=f_cool)
    model_params.add('T_cool', value=T_cool)
    model_params.add('T_phot', value=T_phot)
    model_params.add('R_star', value=R_star)
    model_params.add('conv_sigma', value=conv_sigma)

    # Generate model based on these parameters
    _model = spectral_model(model_params, conv_sigma * u.pixel)
    model_flux = bintogrid(btsettl_wavelengths, _model, newx=data_wave.value)['y']

    # Calculate residuals and log-likelihood
    residuals = data_flux.value - model_flux
    ln_like_spec = -0.5 * np.nansum((residuals / data_err.value) ** 2)

    log_likelihood += ln_like_spec
    return log_likelihood

# Collect all visit data for plotting
all_data = {
    'visit': [],
    'direction': [],
    'wavelength': [],
    'calibrated_flux': [],
    'calibrated_flux_err': [],
    'model_flux': [],
    'residuals': []
}
results_table = []

"""
SET PARAMETERS FOR LABELS AND FILESAVING
"""
ncomponents = 2
speclibrary = 'btsettl'
spectral_model = btsettl_2T
nwalkers = 150
nsteps = 5000

"""Run minimization and MCMC for each visit and direction"""
for visit in ['F21', 'S22']:
    for direction in ['Forward', 'Reverse']:
        print(f"Processing {visit} {direction}\n")

        # Prepare data
        exptime = visits[f'{visit}']['exp (s)']
        binwidth = visits[f'{visit}']['native resolution']
        grism = visits[f'{visit}']['Grism']

        # Load trimmed rainbow object and process the data
        trimmed_r = read_rainbow(f"../data/{visit}_{direction}_trimmed_pacman_spec.rainbow.npy")
        median_spectrum = trimmed_r.get_median_spectrum().value
        e_per_s = median_spectrum / exptime
        e_per_s_per_angstrom = e_per_s / binwidth
        w, s, e = read_sensitivity_curve(grism=grism)
        binned_filter_response = bintogrid(w.value, s.value, newx=trimmed_r.wavelength.value)['y'] * u.cm**2 / u.erg
        calibrated_data_flux = e_per_s_per_angstrom / binned_filter_response
        calibrated_flux_err = 0.005 * calibrated_data_flux

        # Set up lmfit parameters
        parameters = lmfit.Parameters()
        parameters.add('f_cool', value=0.3, min=0.05, max = 0.95)
        parameters.add('T_cool', value=3100, min=2000, max = 3700)
        parameters.add('T_phot', value=4000, min=3000, max=6000, vary=True)
        parameters.add('R_star', value=0.78, min=0.5, max=0.90, vary=True)
        parameters.add('conv_sigma', value=5, min=1.0, max=40.0, vary=True)
        init_sigma = parameters['conv_sigma'].value * u.pixel

        # Run the lmfit minimizer
        result = lmfit.minimize(
            residual, 
            parameters, 
            args=(init_sigma, [trimmed_r.wavelength, calibrated_data_flux, calibrated_flux_err]) )
        
        print(f"Minimization Results for {visit} {direction}:")
        for param_name, param in result.params.items():
            print(f"{param_name}: Value = {param.value:.2f}, 1-sigma = {param.stderr if param.stderr else 'N/A'}")
        
        chi_squared = result.chisqr
        reduced_chi_squared = result.redchi
        dof = result.nfree  # Degrees of freedom
        print(f"Chi-squared: {chi_squared:.2f}")
        print(f"Reduced Chi-squared: {reduced_chi_squared:.2f}")
        print(f"Degrees of freedom: {dof}\n")

        # Append minimization results to the table
        results_entry = {
            'Visit': visit,
            'Direction': direction,
            'Chi-squared': chi_squared,
            'Reduced Chi-squared': reduced_chi_squared
        }
        
        for param_name, param in result.params.items():
            results_entry[f'{param_name} (Value)'] = param.value
            results_entry[f'{param_name} (1-sigma)'] = param.stderr if param.stderr else 'N/A'
        results_table.append(results_entry)

        # Store data for plotting
        all_data['visit'].append(visit)
        all_data['direction'].append(direction)
        all_data['wavelength'].append(trimmed_r.wavelength.value)
        all_data['calibrated_flux'].append(calibrated_data_flux.value)
        all_data['calibrated_flux_err'].append(calibrated_flux_err.value)

        # MCMC setup
        ndim = len(result.params)
        burnin = int(0.5*nsteps)
        initial_pos = [
            np.array([result.params[name].value for name in result.params]) + 1e-4 * np.random.randn(ndim) 
            for _ in range(nwalkers)]
        label=f'{visit}_{direction}_{nsteps}steps_{ncomponents}T_{speclibrary}_specmodel'
        samples_fname = f"../data/samples/{label}.h5"
        backend = emcee.backends.HDFBackend(samples_fname)
        backend.reset(nwalkers, ndim)        
        sampler = emcee.EnsembleSampler(
            nwalkers, ndim, log_probability, 
            args=([trimmed_r.wavelength, calibrated_data_flux, calibrated_flux_err]), 
            backend=backend)

        # Run the MCMC sampler
        print('Running MCMC sampler...')
        sampler.run_mcmc(initial_pos, nsteps, store=True, progress=True)
        samples = sampler.chain[:, burnin:, :].reshape((-1, ndim)).T

        # Check for convergence (all should be > 100)
        for i in range(len(samples)):
            tau_f = emcee.autocorr.integrated_time(samples[i])
            print('(Nsteps-burnin)*nwalkers/tau=',(nsteps-burnin)*nwalkers/tau_f)
        
        # Make corner plot
        rng = 0.9995
        fig = corner.corner( 
            samples.T,show_titles=True, labels=['f_cool','T_cool', 'T_phot', 'R_star', r'$\sigma_{\rm conv}$'],
            range=[rng]*ndim,smooth=1,quantiles=(0.16, 0.5, 0.84),
            fill_contours=True, plot_datapoints=False,title_kwargs={"fontsize": 15},title_fmt='.2f',
            hist_kwargs={"linewidth": 2.5},levels=[(1-np.exp(-0.5)),(1-np.exp(-2)),(1-np.exp(-4.5))]
        )
        plt.suptitle(f"{visit} {direction} MCMC Results")
        plt.savefig(f'../figs/{visit}_{direction}_{nsteps}nsteps_{ncomponents}T_{speclibrary}_corner.png')
        plt.show()

        # Calculate best-fit model flux from MCMC results
        f_cool_sam, T_cool_sam, T_phot_sam, R_star_sam, sigma_sam = samples
        sig1_f_cool = np.percentile(f_cool_sam, [15.9, 50., 84.1]) # central 1-sigma values
        sig1_T_cool = np.percentile(T_cool_sam, [15.9, 50., 84.1]) # central 1-sigma values
        sig1_T_phot = np.percentile(T_phot_sam, [15.9, 50., 84.1]) # central 1-sigma values
        sig1_R_star = np.percentile(R_star_sam, [15.9, 50., 84.1])
        sig1_sigma = np.percentile(sigma_sam, [15.9, 50., 84.1])
        
        # Print the 50th percentile (median) and 1-sigma uncertainties for each parameter
        print("f_cool:")
        print(f"  Median (50th percentile): {sig1_f_cool[1]:.2f}")
        print(f"  Lower 1-sigma uncertainty: {sig1_f_cool[1] - sig1_f_cool[0]:.2f}")
        print(f"  Upper 1-sigma uncertainty: {sig1_f_cool[2] - sig1_f_cool[1]:.2f}")
        print("\nT_cool:")
        print(f"  Median (50th percentile): {sig1_T_cool[1]:.1f}")
        print(f"  Lower 1-sigma uncertainty: {sig1_T_cool[1] - sig1_T_cool[0]:.1f}")
        print(f"  Upper 1-sigma uncertainty: {sig1_T_cool[2] - sig1_T_cool[1]:.1f}")
        print("\nT_phot:")
        print(f"  Median (50th percentile): {sig1_T_phot[1]:.2f}")
        print(f"  Lower 1-sigma uncertainty: {sig1_T_phot[1] - sig1_T_phot[0]:.1f}")
        print(f"  Upper 1-sigma uncertainty: {sig1_T_phot[2] - sig1_T_phot[1]:.1f}")
        print("\nR_star:")
        print(f"  Median (50th percentile): {sig1_R_star[1]:.2f}")
        print(f"  Lower 1-sigma uncertainty: {sig1_R_star[1] - sig1_R_star[0]:.3f}")
        print(f"  Upper 1-sigma uncertainty: {sig1_R_star[2] - sig1_R_star[1]:.3f}")
        print("\nsigma:")
        print(f"  Median (50th percentile): {sig1_sigma[1]:.2f}")
        print(f"  Lower 1-sigma uncertainty: {sig1_sigma[1] - sig1_sigma[0]:.1f}")
        print(f"  Upper 1-sigma uncertainty: {sig1_sigma[2] - sig1_sigma[1]:.1f}")
        
        # Set parameters in lmfit
        mcmc_model_params = lmfit.Parameters()
        mcmc_model_params.add('f_cool', value=sig1_f_cool[1])
        mcmc_model_params.add('T_cool', value=sig1_T_cool[1])
        mcmc_model_params.add('T_phot', value=sig1_T_phot[1])
        mcmc_model_params.add('R_star', value=sig1_R_star[1])
        mcmc_model_params.add('conv_sigma', value=sig1_sigma[1])
        best_sigma = mcmc_model_params['conv_sigma'].value * u.pixel

        # Calculate the max likelihood model
        best_model = spectral_model(mcmc_model_params, best_sigma)
        model_flux_mcmc = bintogrid(btsettl_wavelengths, best_model, newx=trimmed_r.wavelength.value)['y']
        residuals = calibrated_data_flux.value - model_flux_mcmc
        all_data['model_flux'].append(model_flux_mcmc)
        all_data['residuals'].append(residuals)
        
        # Calculate chi-squared and reduced chi-squared
        chisq = np.nansum((residuals / calibrated_flux_err.value) ** 2)
        reduced_chisq = chisq / dof
        
        # Print chi-squared statistics
        print(f"Chi-squared: {chisq:.2f}")
        print(f"Reduced Chi-squared: {reduced_chisq:.2f}")
        print("\n")

In [ ]:
# Range of conv_sigma values for model exploration (5 different values)
conv_sigma_values = [6.0, 12.0]
alphas = [1.0, 0.6]

# Plot all data, models, and residuals on a single plot
fig, (ax, ax_residuals) = plt.subplots(2, 1, gridspec_kw={'height_ratios': [3, 1]}, figsize=(10, 6), sharex=True)
colors = ['blue', 'teal', 'red', 'pink']

for i, (visit, direction, wavelength, data_flux, data_err, model_flux, residual) in enumerate(zip(
    all_data['visit'], all_data['direction'], all_data['wavelength'], all_data['calibrated_flux'], 
    all_data['calibrated_flux_err'], all_data['model_flux'], all_data['residuals'])):
    
    color = colors[i % len(colors)]
    # Plot the observational data with error bars
    ax.errorbar(wavelength, data_flux, yerr=data_err, fmt='o', color=color, ms=1, label=f'{visit} {direction} Data', alpha=0.5)
    # Plot the model using the best-fit parameters
    # ax.plot(wavelength, model_flux, zorder=1000, color='k')
    ax_residuals.plot(wavelength, residual / data_err, color=color, alpha=0.5)

    # Plot models with different conv_sigma values
    for j, sigma in enumerate(conv_sigma_values):
        model_params = lmfit.Parameters()
        model_params.add('f_cool', value=sig1_f_cool[1])
        model_params.add('T_cool', value=sig1_T_cool[1])
        model_params.add('T_phot', value=sig1_T_phot[1])
        model_params.add('R_star', value=sig1_R_star[1])
        model_params.add('conv_sigma', value=sigma)
        # model_params.add('v_shift', value=sig1_v_shift[1])  # Best-fit velocity shift
        
        # Generate model flux for each conv_sigma
        varied_model = spectral_model(model_params, sigma*u.pixel)
        varied_model_flux = bintogrid(btsettl_wavelengths, varied_model, newx=wavelength)['y']
    
        # Plot the varied model flux
        ax.plot(wavelength, varied_model_flux, linestyle='--', label=f'Model (σ={sigma:.1f})', alpha=alphas[j], color='k',zorder=1000)

# Plot configuration
ax.set_ylabel('Flux (erg / s / Å / cm²)')
ax.legend()
ax_residuals.axhline(0, color='black', linestyle='--', linewidth=1)
ax_residuals.set_ylabel('Residuals')
ax_residuals.set_xlabel('Wavelength (μm)')
ax_residuals.legend()

# plt.xlim(0.8, 1.1)

plt.tight_layout()
plt.savefig(f'../figs/model_each_scan_{ncomponents}T_{speclibrary}_resultsplot.png')
plt.show()

# Phoenix 1T Models

In [ ]:
def residual(parameters, sigma, data_list):

    data_wave, data_flux, data_err = data_list
    
    _model = spectral_model(parameters, sigma)
    binned_model = bintogrid(model_wavelengths.value, _model, newx = data_wave.value)
    model_flux = binned_model['y'] * u.erg / u.s / u.angstrom / u.cm / u.cm

    residual = (data_flux-model_flux)/data_err
    
    return residual
    
# Define the log-probability function
def log_probability(parameters, data_wave, data_flux, data_err):
    T_phot, R_star, conv_sigma = parameters

    log_likelihood = 0
    
    # Boundary conditions for parameters to ensure physical validity
    if not (2300 <= T_phot <= 6000 and 0.5 <= R_star <= 0.90 and 0.01 <= conv_sigma <= 50.0):
        return -np.inf  # Return log-probability of -inf outside bounds

    # Set parameters in lmfit
    model_params = lmfit.Parameters()
    model_params.add('T_phot', value=T_phot)
    model_params.add('R_star', value=R_star)
    model_params.add('conv_sigma', value=conv_sigma)

    # Generate model based on these parameters
    _model = spectral_model(model_params, conv_sigma * u.pixel)
    model_flux = bintogrid(model_wavelengths.value, _model, newx=data_wave.value)['y']

    # Calculate residuals and log-likelihood
    residuals = data_flux.value - model_flux
    ln_like_spec = -0.5 * np.nansum((residuals / data_err.value) ** 2)

    log_likelihood += ln_like_spec
    return log_likelihood

# Collect all visit data for plotting
all_data = {
    'visit': [],
    'direction': [],
    'wavelength': [],
    'calibrated_flux': [],
    'calibrated_flux_err': [],
    'model_flux': [],
    'residuals': []
}
results_table = []

"""
SET PARAMETERS FOR LABELS AND FILESAVING
"""
ncomponents = 1
speclibrary = 'phoenix'
spectral_model = phoenix_1T
nwalkers = 100
nsteps = 5000

"""Run minimization and MCMC for each visit and direction"""
for visit in ['F21', 'S22']:
    for direction in ['Forward', 'Reverse']:
        print(f"Processing {visit} {direction}\n")

        # Prepare data
        exptime = visits[f'{visit}']['exp (s)']
        binwidth = visits[f'{visit}']['native resolution']
        grism = visits[f'{visit}']['Grism']

        # Load trimmed rainbow object and process the data
        trimmed_r = read_rainbow(f"../data/{visit}_{direction}_trimmed_pacman_spec.rainbow.npy")
        median_spectrum = trimmed_r.get_median_spectrum().value
        initial_relative_err = 0.01
        e_per_s = median_spectrum / exptime
        e_per_s_per_angstrom = e_per_s / binwidth
        w, s, e = read_sensitivity_curve(grism=grism)
        binned_filter_response = bintogrid(w.value, s.value, newx=trimmed_r.wavelength.value)['y'] * u.cm**2 / u.erg
        calibrated_data_flux = e_per_s_per_angstrom / binned_filter_response
        calibrated_flux_err = initial_relative_err * calibrated_data_flux

        # Set up lmfit parameters
        parameters = lmfit.Parameters()
        parameters.add('T_phot', value=4000, min=2300, max=6000, vary=True)
        parameters.add('R_star', value=0.78, min=0.5, max=0.90, vary=True)
        parameters.add('conv_sigma', value=5, min=0.01, max=50.0, vary=True)
        init_sigma = parameters['conv_sigma'].value * u.pixel

        # Run the lmfit minimizer
        result = lmfit.minimize(
            residual, 
            parameters, 
            args=(init_sigma, [trimmed_r.wavelength, calibrated_data_flux, calibrated_flux_err]) )
        
        print(f"Minimization Results for {visit} {direction}:")
        for param_name, param in result.params.items():
            print(f"{param_name}: Value = {param.value:.2f}, 1-sigma = {param.stderr if param.stderr else 'N/A'}")
        
        chi_squared = result.chisqr
        reduced_chi_squared = result.redchi
        dof = result.nfree  # Degrees of freedom
        print(f"Chi-squared: {chi_squared:.2f}")
        print(f"Reduced Chi-squared: {reduced_chi_squared:.2f}")
        print(f"Degrees of freedom: {dof}\n")

        # Append minimization results to the table
        results_entry = {
            'Visit': visit,
            'Direction': direction,
            'Chi-squared': chi_squared,
            'Reduced Chi-squared': reduced_chi_squared
        }
        
        for param_name, param in result.params.items():
            results_entry[f'{param_name} (Value)'] = param.value
            results_entry[f'{param_name} (1-sigma)'] = param.stderr if param.stderr else 'N/A'
        results_table.append(results_entry)

        # Store data for plotting
        all_data['visit'].append(visit)
        all_data['direction'].append(direction)
        all_data['wavelength'].append(trimmed_r.wavelength.value)
        all_data['calibrated_flux'].append(calibrated_data_flux.value)
        all_data['calibrated_flux_err'].append(calibrated_flux_err.value)

        # MCMC setup
        ndim = len(result.params)
        burnin = int(0.5*nsteps)
        initial_pos = [
            np.array([result.params[name].value for name in result.params]) + 1e-4 * np.random.randn(ndim) 
            for _ in range(nwalkers)]
        label=f'{visit}_{direction}_{nsteps}steps_{ncomponents}T_{speclibrary}_specmodel'
        samples_fname = f"../data/samples/{label}.h5"
        backend = emcee.backends.HDFBackend(samples_fname)
        backend.reset(nwalkers, ndim)        
        sampler = emcee.EnsembleSampler(
            nwalkers, ndim, log_probability, 
            args=([trimmed_r.wavelength, calibrated_data_flux, calibrated_flux_err]), 
            backend=backend)

        # Run the MCMC sampler
        print('Running MCMC sampler...')
        sampler.run_mcmc(initial_pos, nsteps, store=True, progress=True)
        samples = sampler.chain[:, burnin:, :].reshape((-1, ndim)).T

        # Check for convergence (all should be > 100)
        for i in range(len(samples)):
            tau_f = emcee.autocorr.integrated_time(samples[i])
            print('(Nsteps-burnin)*nwalkers/tau=',(nsteps-burnin)*nwalkers/tau_f)
        
        # Make corner plot
        rng = 0.9995
        fig = corner.corner( 
            samples.T,show_titles=True, labels=['T_phot', 'R_star', r'$\sigma_{\rm conv}$'],
            range=[rng]*ndim,smooth=1,quantiles=(0.16, 0.5, 0.84),
            fill_contours=True, plot_datapoints=False,title_kwargs={"fontsize": 15},title_fmt='.2f',
            hist_kwargs={"linewidth": 2.5},levels=[(1-np.exp(-0.5)),(1-np.exp(-2)),(1-np.exp(-4.5))]
        )
        plt.suptitle(f"{visit} {direction} MCMC Results")
        plt.savefig(f'../figs/{visit}_{direction}_{nsteps}nsteps_{speclibrary}_{ncomponents}T_corner.png')
        plt.show()

        # Calculate best-fit model flux from MCMC results
        T_phot_sam, R_star_sam, sigma_sam = samples
        sig1_T_phot = np.percentile(T_phot_sam, [15.9, 50., 84.1]) # central 1-sigma values
        sig1_R_star = np.percentile(R_star_sam, [15.9, 50., 84.1])
        sig1_sigma = np.percentile(sigma_sam, [15.9, 50., 84.1])
        
        # Print the 50th percentile (median) and 1-sigma uncertainties for each parameter
        print("T_phot:")
        print(f"  Median (50th percentile): {sig1_T_phot[1]:.2f}")
        print(f"  Lower 1-sigma uncertainty: {sig1_T_phot[1] - sig1_T_phot[0]:.1f}")
        print(f"  Upper 1-sigma uncertainty: {sig1_T_phot[2] - sig1_T_phot[1]:.1f}")
        print("\nR_star:")
        print(f"  Median (50th percentile): {sig1_R_star[1]:.2f}")
        print(f"  Lower 1-sigma uncertainty: {sig1_R_star[1] - sig1_R_star[0]:.3f}")
        print(f"  Upper 1-sigma uncertainty: {sig1_R_star[2] - sig1_R_star[1]:.3f}")
        print("\nsigma:")
        print(f"  Median (50th percentile): {sig1_sigma[1]:.2f}")
        print(f"  Lower 1-sigma uncertainty: {sig1_sigma[1] - sig1_sigma[0]:.1f}")
        print(f"  Upper 1-sigma uncertainty: {sig1_sigma[2] - sig1_sigma[1]:.1f}")
        
        # Set parameters in lmfit
        mcmc_model_params = lmfit.Parameters()
        mcmc_model_params.add('T_phot', value=sig1_T_phot[1])
        mcmc_model_params.add('R_star', value=sig1_R_star[1])
        mcmc_model_params.add('conv_sigma', value=sig1_sigma[1])
        best_sigma = mcmc_model_params['conv_sigma'].value * u.pixel

        # Calculate the max likelihood model
        best_model = spectral_model(mcmc_model_params, best_sigma)
        model_flux_mcmc = bintogrid(model_wavelengths.value, best_model, newx=trimmed_r.wavelength.value)['y']
        residuals = calibrated_data_flux.value - model_flux_mcmc
        all_data['model_flux'].append(model_flux_mcmc)
        all_data['residuals'].append(residuals)
        
        # Calculate chi-squared and reduced chi-squared
        chisq = np.nansum((residuals / calibrated_flux_err.value) ** 2)
        reduced_chisq = chisq / dof
        
        # Print chi-squared statistics
        print(f"Chi-squared: {chisq:.2f}")
        print(f"Reduced Chi-squared: {reduced_chisq:.2f}")
        print("\n")

# Plot all data, models, and residuals on a single plot
fig, (ax, ax_residuals) = plt.subplots(2, 1, gridspec_kw={'height_ratios': [3, 1]}, figsize=(10, 6), sharex=True)
colors = ['blue', 'teal', 'red', 'pink']

for i, (visit, direction, wavelength, data_flux, data_err, model_flux, residual) in enumerate(zip(
    all_data['visit'], all_data['direction'], all_data['wavelength'], all_data['calibrated_flux'], 
    all_data['calibrated_flux_err'], all_data['model_flux'], all_data['residuals'])):
    
    color = colors[i % len(colors)]
    ax.errorbar(wavelength, data_flux, yerr=data_err, fmt='o', color=color, ms=1, label=f'{visit} {direction} Data', alpha=0.5)
    ax.plot(wavelength, model_flux, zorder=1000 ,color='k')
    ax_residuals.plot(wavelength, residual / data_err, color=color,alpha=0.5)

# Plot configuration
ax.set_ylabel('Flux (erg / s / Å / cm²)')
ax.legend()
ax_residuals.axhline(0, color='black', linestyle='--', linewidth=1)
ax_residuals.set_ylabel('Residuals')
ax_residuals.set_xlabel('Wavelength (μm)')
ax_residuals.legend()

plt.tight_layout()
plt.savefig(f'../figs/model_each_scan_{ncomponents}T_{speclibrary}_resultsplot.png')
plt.show()